<a href="https://colab.research.google.com/github/rayvaril/student-dropout-capstone/blob/main/presentations/06A_Technical_Presentation_Student_Attrition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Early Warning for Student Attrition

## An Explainable Machine Learning Approach with Fairness Evaluation

### Technical Presentation

**Capstone Step 6**

Two-stage early-warning framework:
- Enrollment
- Semester 1

# Data, Target & Success Criteria

## Dataset

**UCI — Predict Students' Dropout and Academic Success**

- **4,424** student records
- **36 predictors + 1 target**
- **0 missing values**
- **0 duplicate records**

## Target

The original outcome has three classes:

- `Dropout`
- `Enrolled`
- `Graduate`

For the main model:

- **1 = Dropout**
- **0 = Not labeled as Dropout (`Enrolled` + `Graduate`)**

> `Enrolled` students are not confirmed eventual non-dropouts, so the binary target must be interpreted carefully.

## Success Criteria

**Primary metric:** Dropout Recall

**Supporting metrics:** Precision, F1, ROC-AUC, PR-AUC

**Operational metric:** Recall@Top10%

# Leakage-Safe Prediction Design

## Enrollment Model

Uses **22 predictors** available at or near enrollment.

Includes:
- demographic and application information
- prior academic information
- macroeconomic context

Excludes:
- Semester 1 academic results
- all Semester 2 variables
- `Debtor`
- `Tuition fees up to date`

The last two are excluded because their exact availability at the enrollment prediction point could not be verified.

## Semester 1 Model

Uses the same **22 Enrollment predictors**, plus:

- **6 raw Semester 1 variables**
- **5 engineered Semester 1 features**

**Total: 33 predictors**

### Engineered Semester 1 Features

- approval rate
- evaluations per enrolled unit
- completion gap
- no-approved-units indicator
- no-enrolled-units indicator

> Feature timing is matched to what would realistically be known when each prediction is made.

# Experimental Design

## Train/Test Split

- **3,539** training students
- **885** held-out test students
- Stratified **80/20** split
- `random_state = 42`

## Model Selection

All model tuning and selection were performed using:

**Stratified 5-fold cross-validation on the training data only**

The held-out test set was reserved exclusively for final evaluation.

## Models Compared

- Logistic Regression
- Random Forest
- Gradient Boosting

## Additional Methods

- Hyperparameter tuning
- L1 feature selection
- PCA dimensionality reduction

> Model selection was based on cross-validation performance—not test-set results.

# Model Selection

## Best Cross-Validated Model

### Tuned Logistic Regression

| Stage | CV Recall | Precision | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
| Enrollment | **70.6%** | 52.4% | 60.1% | 76.9% | 61.4% |
| Semester 1 | **77.1%** | 69.2% | 72.9% | 88.3% | 82.3% |

## Why Logistic Regression?

- Highest Dropout Recall after tuning
- Competitive F1, ROC-AUC, and PR-AUC
- More interpretable than ensemble alternatives
- Supports coefficient-based individual explanations
- Simple and reproducible

> Logistic Regression provided the strongest balance of recall, interpretability, and reproducibility.

# Held-Out Test Performance

| Stage | Recall | Precision | F1 | ROC-AUC | PR-AUC |
|---|---:|---:|---:|---:|---:|
| Enrollment | **72.2%** | 52.6% | 60.8% | 78.6% | 64.5% |
| Semester 1 | **77.5%** | 69.8% | 73.5% | 89.1% | 83.4% |

## Confusion Matrix Outcomes

### Enrollment
- **205** true positives
- **79** false negatives
- **185** false positives
- **416** true negatives

### Semester 1
- **220** true positives
- **64** false negatives
- **95** false positives
- **506** true negatives

## What Improved?

Compared with Enrollment, the Semester 1 model:

- identifies **15 more** students labeled as Dropout
- misses **15 fewer** Dropout cases
- produces **90 fewer** false-positive alerts

> Semester 1 improves both dropout detection and targeting efficiency.

# Explainability: What Drives the Models?

## Enrollment Model

The strongest permutation-importance signals include:

- scholarship status
- course
- application mode
- gender
- admission grade
- age at enrollment

## Semester 1 Model

Once academic-progress data becomes available, the strongest signals shift toward:

- curricular units approved
- Semester 1 grade
- curricular units enrolled
- approval rate
- no-approved-units indicator
- completion gap

## Partial Dependence Finding

Predicted dropout risk generally decreases as:

- the number of approved Semester 1 units increases
- Semester 1 grades improve

> The later model relies more heavily on observed academic progress, while the earlier model depends more on background and enrollment-stage characteristics.

**Important:** These are predictive relationships, not causal effects.

# Fairness Audit

Fairness was evaluated on the held-out test set across:

- Gender
- Age groups
- Scholarship status

## Dropout Recall Gaps

| Group | Enrollment | Semester 1 |
|---|---:|---:|
| Gender | **29.1 pts** | **9.7 pts** |
| Age | **45.4 pts** | **27.7 pts** |
| Scholarship status | **65.0 pts** | **40.2 pts** |

## Equalized-Odds Gap

| Group | Enrollment | Semester 1 |
|---|---:|---:|
| Gender | 0.340 | **0.108** |
| Age | 0.544 | **0.277** |
| Scholarship status | 0.650 | **0.402** |

## Interpretation

Semester 1 information reduces every observed subgroup gap, but does not eliminate them.

The largest remaining disparities are across:

- age groups
- scholarship status

> These results do not prove the model is fair or unfair. They identify subgroup performance differences that require monitoring and mitigation before deployment.

# Fairness Audit

Fairness was evaluated on the held-out test set across:

- Gender
- Age groups
- Scholarship status

## Dropout Recall Gaps

| Group | Enrollment | Semester 1 |
|---|---:|---:|
| Gender | **29.1 pts** | **9.7 pts** |
| Age | **45.4 pts** | **27.7 pts** |
| Scholarship status | **65.0 pts** | **40.2 pts** |

## Equalized-Odds Gap

| Group | Enrollment | Semester 1 |
|---|---:|---:|
| Gender | 0.340 | **0.108** |
| Age | 0.544 | **0.277** |
| Scholarship status | 0.650 | **0.402** |

## Interpretation

Semester 1 information reduces every observed subgroup gap, but does not eliminate them.

The largest remaining disparities are across:

- age groups
- scholarship status

> These results do not prove the model is fair or unfair. They identify subgroup performance differences that require monitoring and mitigation before deployment.

# Conclusions, Limitations & Next Steps

## Conclusions

- Dropout risk can already be identified at Enrollment
- Semester 1 substantially improves predictive performance
- Later predictions rely more on observed academic progress
- Semester 1 also reduces several observed fairness gaps
- A two-stage approach provides both earlier warning and stronger later targeting

## Limitations

- Single dataset and institutional context
- `Enrolled` students are included in the 0 class
- Predictive relationships are not causal
- Some fairness subgroups have limited positive-case counts
- Future-cohort performance is unknown

## Before Deployment

1. Validate on future student cohorts
2. Define intervention capacity and operating thresholds
3. Test fairness-mitigation strategies
4. Monitor subgroup Recall and false-negative rates
5. Maintain human review for intervention decisions

## Current Status

**Promising decision-support prototype — not deployment-ready**

> Use the model to support student outreach, not to automate decisions about students.